# Nemotron Reasoning Challenge — Kaggle Debug (Qwen 7B)

Same pipeline as the main notebook but uses **Qwen/Qwen2.5-7B-Instruct** instead of the 30B Nemotron. This model fits comfortably on Kaggle T4 / T4x2 and trains in ~1-2 hours.

Use this to validate the full pipeline (EDA, data prep, LoRA training, packaging). For the actual competition submission, train the Nemotron adapter on a larger machine.

In [ ]:
from pathlib import Path
import os

IS_KAGGLE = os.path.exists("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/project" if IS_KAGGLE else ".").resolve()

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
HF_TOKEN = os.environ.get("HF_TOKEN", "")

GPU_PROFILE = "t4"
USE_BF16_FULL = False
TRAIN_MAX_SEQ = 2048
TRAIN_MAX_MEMORY_JSON = None  # 7B model fits in a single T4 without max_memory tricks

LORA_TARGET_MODE = "hf_linear"
LORA_ALPHA = 32
TRAIN_BATCH = 2
GRAD_ACCUM = 8
NUM_EPOCHS = 2.0

SKIP_COT = True
SYNTHETIC_PER_KIND = 400
RUN_VLLM_EVAL = False

print("IS_KAGGLE:", IS_KAGGLE)
print("WORK_ROOT:", WORK_ROOT)
print("MODEL_ID:", MODEL_ID)
print("GPU_PROFILE:", GPU_PROFILE, "| TRAIN_MAX_SEQ:", TRAIN_MAX_SEQ)

## Install dependencies

In [ ]:
import subprocess, sys

def pip_install(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

def pip_try(*args: str) -> bool:
    try:
        pip_install(*args)
        return True
    except subprocess.CalledProcessError:
        return False

pip_install("-U", "pip", "setuptools", "wheel")
pip_install(
    "transformers>=4.45,<5",
    "peft>=0.12",
    "trl>=0.12",
    "datasets",
    "accelerate",
    "bitsandbytes",
    "psutil",
    "pandas",
    "numpy",
    "scikit-learn",
    "tqdm",
    "huggingface_hub",
)

if not pip_try("polars"):
    print("polars install skipped (optional).")

print("\n--- Installed versions ---")
import torch
for pkg in ("torch", "transformers", "peft", "trl", "bitsandbytes"):
    try:
        v = __import__(pkg).__version__
        print(f"  {pkg}: {v}")
    except Exception:
        print(f"  {pkg}: not found")

## Setup workspace and data

In [ ]:
import os, sys, shutil
from pathlib import Path

WORK_ROOT.mkdir(parents=True, exist_ok=True)
data_dir = WORK_ROOT / "data"
data_dir.mkdir(parents=True, exist_ok=True)
(WORK_ROOT / "data" / "reports").mkdir(parents=True, exist_ok=True)
(WORK_ROOT / "data" / "synthetic").mkdir(parents=True, exist_ok=True)

if IS_KAGGLE:
    _kaggle_root = Path("/kaggle/input")
    _code_source = None
    for d in sorted(_kaggle_root.iterdir()):
        if d.is_dir() and (d / "scripts" / "01_eda.py").is_file():
            _code_source = d
            break
    if _code_source is None:
        try:
            for eda in _kaggle_root.rglob("01_eda.py"):
                if eda.parent.name == "scripts":
                    _code_source = eda.parent.parent.resolve()
                    break
        except OSError:
            pass
    if _code_source is not None:
        for item in ("scripts", "requirements.txt"):
            src = _code_source / item
            if not src.exists():
                continue
            dst = WORK_ROOT / item
            if dst.exists():
                shutil.rmtree(dst) if dst.is_dir() else dst.unlink()
            if src.is_dir():
                shutil.copytree(src, dst)
            else:
                shutil.copy2(src, dst)
        print("Copied code from", _code_source)

    _comp_dirs = [
        _kaggle_root / "competitions" / "nvidia-nemotron-model-reasoning-challenge",
        _kaggle_root / "nvidia-nemotron-model-reasoning-challenge",
    ]
    _comp_data = None
    for cd in _comp_dirs:
        if cd.is_dir() and (cd / "train.csv").is_file():
            _comp_data = cd
            break
    if _comp_data is None:
        for d in sorted(_kaggle_root.iterdir()):
            if d.is_dir() and (d / "train.csv").is_file():
                _comp_data = d
                break
    if _comp_data is not None:
        for fname in ("train.csv", "test.csv"):
            src = _comp_data / fname
            if src.is_file():
                shutil.copy2(src, data_dir / fname)
                print("Copied", fname)

os.chdir(WORK_ROOT)
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

assert (WORK_ROOT / "scripts" / "01_eda.py").is_file()
assert (data_dir / "train.csv").is_file()
print("cwd:", os.getcwd())
print("train.csv:", (data_dir / "train.csv").stat().st_size, "bytes")

## Download model

In [ ]:
from huggingface_hub import login, snapshot_download

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("HF login: OK")
else:
    try:
        login(add_to_git_credential=False)
        print("HF login: OK (interactive)")
    except Exception as e:
        print("HF login skipped:", e)

MODEL_PATH_LOCAL = snapshot_download(MODEL_ID, resume_download=True)
print("Model cached at:", MODEL_PATH_LOCAL)

## Phase 1 — EDA

In [ ]:
import subprocess, sys

subprocess.run(
    [
        sys.executable,
        "scripts/01_eda.py",
        "--data-dir", "data",
        "--report-dir", "data/reports",
        "--tokenizer-model", str(MODEL_PATH_LOCAL),
    ],
    check=True,
)

## Phase 2 — Prepare SFT data

In [ ]:
import subprocess, sys

skip = ["--skip-cot"] if SKIP_COT else []
cmd = [
    sys.executable,
    "scripts/02_prepare_data.py",
    "--data-dir", "data",
    "--synthetic-dir", "data/synthetic",
    "--output", "data/train_sft.jsonl",
    "--tokenizer-model", str(MODEL_PATH_LOCAL),
    "--synthetic-per-kind", str(SYNTHETIC_PER_KIND),
] + skip
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## Phase 3 — LoRA training (Qwen 7B)

In [ ]:
import gc, os, subprocess, sys

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

train_env = os.environ.copy()
for k, v in (
    ("OMP_NUM_THREADS", "1"),
    ("MKL_NUM_THREADS", "1"),
    ("TOKENIZERS_PARALLELISM", "false"),
    ("NEMOTRON_KAGGLE_PATCHES", "0"),
):
    train_env.setdefault(k, v)

cmd = [
    sys.executable,
    "scripts/03_train_lora.py",
    "--data-path", "data/train_sft.jsonl",
    "--output-dir", "lora_adapter",
    "--checkpoint-dir", "lora_output",
    "--model-path", str(MODEL_PATH_LOCAL),
    "--lora-target-mode", LORA_TARGET_MODE,
    "--lora-alpha", str(LORA_ALPHA),
    "--batch-size", str(TRAIN_BATCH),
    "--grad-accum", str(GRAD_ACCUM),
    "--epochs", str(NUM_EPOCHS),
    "--max-seq-length", str(TRAIN_MAX_SEQ),
    "--force-peft",
    "--dataloader-workers", "0",
    "--no-nemotron-kaggle-patches",
    "--adapter-base-name", MODEL_ID,
]

gc.collect()
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=train_env)

## Phase 5 — Package submission

In [ ]:
import subprocess, sys

subprocess.run(
    [
        sys.executable,
        "scripts/05_package_submission.py",
        "--adapter-dir", "lora_adapter",
        "--output", "submission.zip",
    ],
    check=True,
)

zp = WORK_ROOT / "submission.zip"
print("submission.zip:", zp.is_file(), zp.stat().st_size if zp.is_file() else 0, "bytes")

if IS_KAGGLE:
    from IPython.display import FileLink, display
    display(FileLink("submission.zip"))